In [ ]:
import matplotlib.animation as animation
from matplotlib.colors import Normalize
import matplotlib.pyplot as plt
import rasterio
import contextily as ctx
import numpy as np

haz_maps_names = create_dict_haz_files(hazard_path)


# Get unique timesteps from the results
unique_timesteps = sorted(df_results['timestep'].unique())
print(f"Creating animation for {len(unique_timesteps)} timesteps")

# Set up the figure and axis
fig, ax = plt.subplots(figsize=(12, 10))

def animate(frame):
    ax.clear()
    
    timestep = unique_timesteps[frame]
    
    # Open the hazard map for the corresponding day
    day = timestep // 24
    hazard_map_path = hazard_path / haz_maps_names[day]
    
    if hazard_map_path.exists():
        with rasterio.open(hazard_map_path) as src:
            data = src.read(1)
            
            # Create mask for areas with significant hazard values (flooding)
            flooding_mask = (data > 0.01) & (data != src.nodata)
            
            # Simple approach: set non-flooding areas to NaN
            flood_data = np.where(flooding_mask, data, np.nan)
            
            # Plot flood data - NaN will be transparent
            if np.any(flooding_mask):  # Only plot if there's flooding data
                im = ax.imshow(flood_data, cmap='Blues', vmin=0.01, vmax=0.5,
                              extent=[src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top],
                              origin='upper', alpha=0.9)
    
    # Get data for current timestep
    current_timestep_data = df_results[df_results['timestep'] == timestep]
    
    # Convert to lat/lon for plotting
    gdf_for_plot = gdf_asset_geometries.copy()
    if hazard_map_path.exists():
        with rasterio.open(hazard_map_path) as src:
            gdf_for_plot = gdf_for_plot.to_crs('EPSG:4326')
    
    # Plot assets in order: operational first (background), then special states on top
    
    # 1. Plot inaccessible assets (orange) - these can still be operational but if they are inaccessible and operational, we care mroe about operational
    inaccessible_assets = current_timestep_data[~current_timestep_data['accessible']]
    if len(inaccessible_assets) > 0:
        inaccessible_geometries = gdf_for_plot.iloc[inaccessible_assets['asset_id']].copy()
        inaccessible_geometries.plot(ax=ax, color='orange', markersize=15, alpha=1, label='Inaccessible')    

    # 2. Plot all operational assets (green)
    operational_assets = current_timestep_data[current_timestep_data['operational']]
    if len(operational_assets) > 0:
        operational_geometries = gdf_for_plot.iloc[operational_assets['asset_id']].copy()
        operational_geometries.plot(ax=ax, color='green', markersize=15, alpha=1, label='Operational')
    
    # 3. Plot non-operational assets (red) - highest priority, plotted last
    non_operational_assets = current_timestep_data[~current_timestep_data['operational']]
    if len(non_operational_assets) > 0:
        non_op_geometries = gdf_for_plot.iloc[non_operational_assets['asset_id']].copy()
        non_op_geometries.plot(ax=ax, color='red', markersize=25, alpha=1, label='Non-operational')    
        
    # 4. Plot assets under repair (yellow) - these are non-operational but being fixed
    under_repair_assets = current_timestep_data[current_timestep_data['under_repair']]
    if len(under_repair_assets) > 0:
        under_repair_geometries = gdf_for_plot.iloc[under_repair_assets['asset_id']].copy()
        under_repair_geometries.plot(ax=ax, color='yellow', markersize=20, alpha=1, label='Under Repair')
    
    # Add basemap (after plotting points to ensure proper extent)
    if hazard_map_path.exists():
        ctx.add_basemap(ax, crs='EPSG:4326', source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.5)
    
    # Add timestep and day indicator
    day_info = f"Timestep: {timestep} (Day {day})"
    ax.text(0.05, 0.95, day_info, transform=ax.transAxes, fontsize=14,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', boxstyle='round,pad=0.5'))
    
    # Add status summary
    total_assets = len(current_timestep_data)
    operational_count = len(operational_assets)
    non_operational_count = len(non_operational_assets)
    under_repair_count = len(under_repair_assets)
    inaccessible_count = len(inaccessible_assets)
    
    status_text = f"Operational: {operational_count}/{total_assets} ({operational_count/total_assets*100:.1f}%)\n"
    status_text += f"Non-operational: {non_operational_count}\n"
    status_text += f"Under repair: {under_repair_count}\n"
    status_text += f"Inaccessible: {inaccessible_count}"
    
    ax.text(0.05, 0.25, status_text, transform=ax.transAxes, fontsize=10,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', boxstyle='round,pad=0.5'))
    
    # Set labels and title
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Asset Status Over Time - Flood Impact and Recovery')
    
    # Set the x and y limits to the extent of the substations
    bounds = gdf_for_plot.total_bounds
    margin = 0.01  # Add some margin
    ax.set_xlim(bounds[0] - margin, bounds[2] + margin)
    ax.set_ylim(bounds[1] - margin, bounds[3] + margin)
    
    # Add static legend with all possible elements
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=8, alpha=1, label='Operational'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, alpha=1, label='Non-operational'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='yellow', markersize=9, alpha=1, label='Under Repair'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', markersize=8, alpha=1, label='Inaccessible')
    ]
    ax.legend(handles=legend_elements, loc='upper right')

# Create animation with slower interval
print("Creating animation...")
anim = animation.FuncAnimation(fig, animate, frames=len(unique_timesteps), 
                              interval=1000, repeat=True, blit=False)

# Save animation to file with full path
print("Saving animation...")
output_path = root_dir / 'data' / 'output' / 'flood_animation_2.gif'
if not output_path.parent.exists():
    output_path.parent.mkdir(parents=True, exist_ok=True)

anim.save(str(output_path), writer='pillow', fps=6)
print(f"Animation saved to: {output_path}")

# Display the animation
plt.tight_layout()
plt.show()
